In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 8.4/8.4 MB 45.0 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ------------------------ --------------- 22.8/37.4 MB 116.6 MB/s eta 0:00:01
   -------------------------------- ------- 30.7/37.4 MB 74.0 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.4 MB 61.7 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 53.4 MB/s  0:00:00

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------

In [5]:
pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
df = pd.read_csv(r"d:\BUU\python\ecommerce_sales_analytics_5000.csv")
print(df)

      order_id order_date  customer_id product_category region  quantity  \
0        10001   1/1/2022         1102           Beauty  South         7   
1        10002   1/2/2022         1435         Clothing  South         7   
2        10003   1/3/2022         1860           Beauty   East         3   
3        10004   1/4/2022         1270      Electronics   West         5   
4        10005   1/5/2022         1106         Clothing   West         5   
...        ...        ...          ...              ...    ...       ...   
4995     14996   9/5/2035         1289         Clothing   West         2   
4996     14997   9/6/2035         1294           Beauty  North         3   
4997     14998   9/7/2035         1450           Beauty   West         3   
4998     14999   9/8/2035         1903      Electronics  South         4   
4999     15000   9/9/2035         1214             Home   East         7   

      unit_price  discount payment_method  delivery_days  customer_rating  \
0         

In [9]:
import os
import sys

import matplotlib

matplotlib.use("Agg")  # ลบบรรทัดนี้ถ้าอยากให้แสดงหน้าต่างกราฟ แล้วเปลี่ยน savefig เป็น plt.show()
import matplotlib.pyplot as plt
import matplotlib.ticker
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

# ---------- ตำแหน่งไฟล์ ----------
CSV_PATH = r"d:\BUU\python\ecommerce_sales_analytics_5000.csv"

# กรอง argument ที่มาจาก Jupyter / IPykernel (ที่ขึ้นต้นด้วย '-') ออกไป
args = [a for a in sys.argv[1:] if not a.startswith("-")]

CSV = args[0] if len(args) > 0 else CSV_PATH
OUT_DIR = args[1] if len(args) > 1 else os.path.join(os.path.dirname(CSV), "charts_5")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- สี (ไล่สีม่วงจากอ่อนไปเข้ม) ----------
LIGHT = "#E6DFFA"        # ม่วงอ่อนสุด
DARK = "#4B3A9A"         # ม่วงเข้มสุด
EDGE = "#3B2D7A"         # สีขอบ
LINE = "#E05A67"         # เส้นตรง Least Squares (แดงชมพู)
GREY = "#A8A8A8"
TEXT = "#333333"
BG = "#FAFAFA"
cmap = LinearSegmentedColormap.from_list("light_to_dark_purple", [LIGHT, "#B19FF1", DARK])


def shade(n, lo=0.15, hi=1.0):
    """คืนลิสต์สี n สี ไล่จากอ่อนไปเข้ม"""
    return [cmap(v) for v in np.linspace(lo, hi, n)]


comma = matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:,.0f}")


def new_figure(title, xlabel, ylabel, figsize=(10, 6), grid=True):
    plt.figure(figsize=figsize, facecolor=BG)
    ax = plt.axes()
    ax.set_facecolor("#FFFFFF")
    plt.title(title, fontsize=14, fontweight="bold", color=TEXT, pad=15)
    plt.xlabel(xlabel, fontsize=12, labelpad=10)
    plt.ylabel(ylabel, fontsize=12, labelpad=10)
    if grid:
        plt.grid(axis="y", linestyle=":", alpha=0.6, zorder=1)
    return ax


def save(name, tight_bbox=False):
    plt.tight_layout()
    path = os.path.join(OUT_DIR, name)
    kwargs = {"bbox_inches": "tight"} if tight_bbox else {}
    plt.savefig(path, dpi=150, facecolor=plt.gcf().get_facecolor(), **kwargs)
    plt.close()
    print("saved", path)


# ---------- โหลดข้อมูล ----------
df = pd.read_csv(CSV, encoding="utf-8-sig")[["quantity", "revenue"]]
qs = sorted(df["quantity"].unique())
qmin, qmax = qs[0], qs[-1]

# เส้นตรง Least Squares จากข้อมูลทั้งหมด
m, c = np.polyfit(df["quantity"], df["revenue"], 1)
r = df["quantity"].corr(df["revenue"])
x_line = np.linspace(qmin, qmax, 100)

# ============ 1) Linear (ปกติ): จุดข้อมูลดิบ + เส้นตรง + drop lines ============
sample = df.sample(n=40, random_state=42).sort_values("quantity").reset_index(drop=True)
xs, ys = sample["quantity"].values, sample["revenue"].values
fitted = m * xs + c

ax = new_figure("1) Linear Regression: Quantity vs Revenue", "Quantity", "Revenue")
ax.yaxis.set_major_formatter(comma)
plt.plot(x_line, m * x_line + c, color=LINE, linewidth=3,
         label=f"Least Squares Line (r = {r:.2f})", zorder=2)
for i in range(len(sample)):
    plt.vlines(xs[i], min(ys[i], fitted[i]), max(ys[i], fitted[i]),
               colors=GREY, linestyles="--", linewidth=1.2, zorder=3)
sc = plt.scatter(xs, ys, c=ys, cmap=cmap, s=60, edgecolors=EDGE, linewidths=1,
                 zorder=4, label="Data Points (random sample, n=40)")
plt.colorbar(sc, label="Revenue", format=comma, pad=0.02)
plt.xticks(qs)
plt.legend(frameon=True, facecolor="#FFFFFF", edgecolor="none", loc="upper left")
save("1_linear_regression.png")

# ============ 2) Linear (เฉลี่ย): ค่าเฉลี่ยของแต่ละ quantity + เส้นตรง ============
mean_rev = df.groupby("quantity")["revenue"].mean()

ax = new_figure("2) Linear Regression: Average Revenue by Quantity", "Quantity",
                "Average Revenue per Order")
ax.yaxis.set_major_formatter(comma)
plt.plot(x_line, m * x_line + c, color=LINE, linewidth=3,
         label="Least Squares Line (all orders)", zorder=2)
plt.scatter(mean_rev.index, mean_rev.values, c=shade(len(mean_rev)), s=130,
            edgecolors=EDGE, linewidths=1.2, zorder=4, label="Average per Quantity")
for q, v in mean_rev.items():
    plt.text(q, v + 70, f"{v:,.0f}", ha="center", fontsize=10, color=TEXT)
plt.ylim(0, mean_rev.max() * 1.18)
plt.xticks(qs)
plt.legend(frameon=True, facecolor="#FFFFFF", edgecolor="none", loc="upper left")
save("2_linear_average.png")

# ============ 3) Bar: รายได้รวมตาม quantity ============
total_rev = df.groupby("quantity")["revenue"].sum()

ax = new_figure("3) Total Revenue by Quantity", "Quantity", "Total Revenue")
ax.yaxis.set_major_formatter(comma)
bars = plt.bar(total_rev.index.astype(str), total_rev.values, color=shade(len(total_rev)),
               edgecolor=EDGE, linewidth=1, zorder=3)
for b, v in zip(bars, total_rev.values):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v:,.0f}", ha="center",
             va="bottom", fontsize=10, color=TEXT)
plt.ylim(0, total_rev.max() * 1.1)
save("3_total_revenue_bar.png")

# ============ 4) Boxplot: การกระจาย revenue ตาม quantity (พร้อมบอกตัวเลขแต่ละค่า) ============
data = [df.loc[df["quantity"] == q, "revenue"] for q in qs]

ax = new_figure("4) Revenue Distribution by Quantity (Boxplot)", "Quantity", "Revenue per Order")
ax.yaxis.set_major_formatter(comma)

bp = plt.boxplot(data, label=qs, patch_artist=True, widths=0.45,
                 medianprops=dict(color=LINE, linewidth=2.5),
                 whiskerprops=dict(color=GREY, linewidth=1.2),
                 capprops=dict(color=GREY, linewidth=1.2),
                 flierprops=dict(marker="o", markersize=3, markerfacecolor=GREY,
                                 markeredgecolor=GREY, alpha=0.5))

for box, col in zip(bp["boxes"], shade(len(qs))):
    box.set(facecolor=col, edgecolor=EDGE, linewidth=1)

# เพิ่มตัวเลขระบุค่าสถิติ (5-Number Summary) ประจำแต่ละ quantity
for idx, q in enumerate(qs, start=1):
    q_data = df.loc[df["quantity"] == q, "revenue"]
    q1 = q_data.quantile(0.25)
    med = q_data.median()
    q3 = q_data.quantile(0.75)
    iqr = q3 - q1
    
    whis_low = q_data[q_data >= q1 - 1.5 * iqr].min()
    whis_high = q_data[q_data <= q3 + 1.5 * iqr].max()

    offset_x = idx + 0.25  # ระยะเว้นข้อความขยับไปทางขวาของกล่อง
    
    # แสดงค่า Max, Q3, Median, Q1, Min
    ax.text(offset_x, whis_high, f"Max: {whis_high:,.0f}", va="center", fontsize=8, color="#555555")
    ax.text(offset_x, q3, f"Q3: {q3:,.0f}", va="center", fontsize=8, color=TEXT)
    ax.text(offset_x, med, f"Med: {med:,.0f}", va="center", fontsize=8, fontweight="bold", color=LINE)
    ax.text(offset_x, q1, f"Q1: {q1:,.0f}", va="center", fontsize=8, color=TEXT)
    ax.text(offset_x, whis_low, f"Min: {whis_low:,.0f}", va="center", fontsize=8, color="#555555")

plt.xlim(0.5, len(qs) + 0.8)
save("4_boxplot.png")

# ============ 5) Donut: สัดส่วนรายได้รวมของแต่ละ quantity ============
total = total_rev.sum()
share = total_rev / total * 100
colors = shade(len(total_rev))

fig, ax = plt.subplots(figsize=(9, 7), facecolor=BG)
wedges, _ = ax.pie(
    total_rev.values,
    colors=colors,
    startangle=90,
    counterclock=False,
    wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2),
)
for w, pct, col in zip(wedges, share.values, colors):
    ang = np.deg2rad((w.theta1 + w.theta2) / 2)
    rad = 1 - 0.42 / 2
    lum = 0.299 * col[0] + 0.587 * col[1] + 0.114 * col[2]
    ax.text(rad * np.cos(ang), rad * np.sin(ang), f"{pct:.1f}%", ha="center", va="center",
            fontsize=11, fontweight="bold", color="white" if lum < 0.55 else TEXT)

ax.text(0, 0.07, f"{total:,.0f}", ha="center", va="center", fontsize=22,
        fontweight="bold", color=TEXT)
ax.text(0, -0.1, "Total Revenue", ha="center", va="center", fontsize=12, color="#666666")
ax.set_title("5) Revenue Share by Quantity (Donut)", fontsize=14, fontweight="bold",
             color=TEXT, pad=20)
ax.legend(wedges, [f"Quantity {q}  ({v:,.0f})" for q, v in total_rev.items()],
          loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False, fontsize=11)

save("5_donut.png", tight_bbox=True)

saved d:\BUU\python\charts_5\1_linear_regression.png
saved d:\BUU\python\charts_5\2_linear_average.png
saved d:\BUU\python\charts_5\3_total_revenue_bar.png
saved d:\BUU\python\charts_5\4_boxplot.png
saved d:\BUU\python\charts_5\5_donut.png
